In [2]:
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


c:\Users\agsto\Desktop\CMPE-252-summary-project\summary_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer,DataCollatorForSeq2Seq, Seq2SeqTrainingArguments
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())       # True if GPU is usable
print("CUDA version:", torch.version.cuda)                # CUDA version PyTorch was built with
print("Number of GPUs:", torch.cuda.device_count())       # Number of GPUs detected
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4070 SUPER


## Importing and cleaning data set

In [3]:
documention_path_0 = "document_train_0.parquet" #file path
document_0 = pd.read_parquet(documention_path_0)

documention_path_1 = "document_train_1.parquet"
document_1 = pd.read_parquet(documention_path_1)

documention_path_2 = "document_train_2.parquet"
document_2 = pd.read_parquet(documention_path_2)

documention_path_3 = "document_train_3.parquet"
document_3 = pd.read_parquet(documention_path_3)

In [4]:
document = pd.concat([document_0, document_1, document_2, document_3])

In [5]:
document

,article,abstract
0,additive models @xcite provide an important fa...,additive models play an important role in semi...
1,the leptonic decays of a charged pseudoscalar ...,"we have studied the leptonic decay @xmath0 , v..."
2,the transport properties of nonlinear non - eq...,"in 84 , 258 ( 2000 ) , mateos conjectured that..."
3,studies of laser beams propagating through tur...,the effect of a random phase diffuser on fluct...
4,the so - called `` nucleon spin crisis '' rais...,with a special intention of clarifying the und...
...,...,...
13531,there has been a recent upsurge in interest in...,we present measurements of the integrated flux...
13532,the last few years have seen growing evidence ...,we show how a very accurate measurement of the...
13533,"the observed initial mass function of stars , ...",we demonstrate the feasibility of detecting di...
13534,numerical simulations of the growth of large s...,a clustering analysis is performed on two samp...


In [233]:
def clean_document(document):
    document['abstract'] = document['abstract'].str.replace(' \n', '', regex=False) #remove \n
    document['abstract'] = document['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['abstract'] = document['abstract'].str.replace('  ', ' ', regex=False) #replace double space with single space

    document['article'] = document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['article'] = document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

    article_summary = document['article'].str.len().describe()
    abstract_summary = document['abstract'].str.len().describe()

    document = document[document['article'].str.len() >= article_summary['25%']] # has to be at least 25% article length
    document = document[document['article'].str.len() <= article_summary['75%']] # has to be at most 75% article length

    document = document[document['abstract'].str.len() >= abstract_summary['25%']] 
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document[document['article'].str.len() >= article_summary['25%']] 
    document = document[document['article'].str.len() <= article_summary['75%']]  

    document = document[document['abstract'].str.len() >= abstract_summary['25%']]
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document.drop_duplicates()
    document = document.reset_index(drop=True)
    
    return document

In [7]:
document = clean_document(document)

## Validation and Test cleaning

In [8]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True, torch_dtype = torch.bfloat16)

Please make sure the generation config includes `forced_bos_token_id=0`. 
The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 667.97it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [4]:
validation_path = "document_validation.parquet" #file path
validation = pd.read_parquet(validation_path)

test_path = "document_test.parquet"
test = pd.read_parquet(test_path)

In [5]:
validation = clean_document(validation)

In [35]:
test = clean_document(test)

## Training

In [9]:
train_dataset = Dataset.from_pandas(document)
validation_dataset   = Dataset.from_pandas(validation)
test_dataset  = Dataset.from_pandas(test)

NameError: name 'validation' is not defined

In [ ]:
label = tokenizer(
    list(train_dataset["abstract"]),
    truncation=True,
    max_length=1024,
    padding=False
)

# Get lengths
label_lengths = [len(ids) for ids in label["input_ids"]]

In [ ]:
abstract_tokens_summary = pd.DataFrame(label_lengths).describe()[0]
abstract_tokens_summary

count    15490.000000
mean       193.410781
std         42.827134
min        106.000000
25%        159.000000
50%        189.000000
75%        223.000000
max        544.000000
Name: 0, dtype: float64

In [ ]:
token_len_IQR =  abstract_tokens_summary['75%'] - abstract_tokens_summary['25%']
1.5 * token_len_IQR

np.float64(96.0)

In [ ]:
abstract_tokens_summary['25%'] - (1.5 * token_len_IQR)

np.float64(63.0)

In [ ]:
abstract_tokens_summary['75%'] + (1.5 * token_len_IQR)

#pad to 320 should be a clean number

np.float64(319.0)

In [ ]:
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").cuda()
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1132.74it/s, Materializing param=model.shared.weight]                                   


In [ ]:
def tokenize_document(dataset):
    model_input = tokenizer(dataset["article"], max_length = 1024, truncation = True)
    labels = tokenizer(dataset["abstract"], max_length = 320, truncation = True)

    model_input["labels"] = labels['input_ids']
    return model_input

In [ ]:
tokenized_dataset = train_dataset.map(tokenize_document, batched=True)
validation_tokenized = validation_dataset.map(tokenize_document, batched=True, remove_columns=validation_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_document, batched=True, remove_columns=test_dataset.column_names)

Map: 100%|██████████| 1715/1715 [00:05<00:00, 298.92 examples/s]


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest"  # pads to the longest sequence in each batch
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",      # folder to save checkpoints
    num_train_epochs=3,                  # number of passes through the dataset
    per_device_train_batch_size=4,       # safe for 12GB VRAM
    per_device_eval_batch_size=4,        # same as train batch size
    learning_rate=2e-5,                  # learning rate
    weight_decay=0.01,                   # regularization
    save_total_limit=3,                   # keep last 3 checkpoints
    eval_strategy="epoch",          # evaluate once per epoch
    save_strategy="epoch",                # save checkpoint once per epoch
    logging_strategy="steps",             # log every N steps
    logging_steps=100,                    # adjust if needed
    bf16=True,                            # mixed precision for VRAM efficiency
    predict_with_generate=True            # necessary for seq2seq evaluation
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint="./bart_finetuned/checkpoint-7746") #pick up from Epoch = 2

### Epoch	Training Loss	Validation Loss
### 1	2.795941	2.686332
### 2	2.850331	2.683255
### 3	2.777663	2.682559

## ROUGE

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If model returns tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )

    # Replace -100 in labels (ignore index) so they can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

## Trained model ROUGE

In [ ]:
#epoch = 3

rouge = evaluate.load("rouge")

predictions = trainer.predict(validation_tokenized)

preds = predictions.predictions
labels = predictions.label_ids

labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
)

print({k: round(v * 100, 4) for k, v in results.items()})

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}


### ROUGE results from epoch = 2

{'rouge1': np.float64(44.6586), 'rouge2': np.float64(15.9589), 'rougeL': np.float64(24.7595), 'rougeLsum': np.float64(24.7636)}

### ROUGE results from epoch = 3

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}

# Summary Generation comparison

In [4]:
train_section = pd.read_parquet('section_train_4.parquet')

train_section['article'] = train_section['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
train_section['article'] = train_section['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

train_document = pd.read_parquet('document_train_4.parquet')

train_document['article'] = train_document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
train_document['article'] = train_document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

In [18]:
test_ind = 3

In [19]:
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746", attn_implementation="eager").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

inputs = tokenizer(
    train_document['article'][test_ind],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1193.48it/s, Materializing param=model.shared.weight]                                   


a simple analytic approximation formula for the bond price in the chan - karolyi - longstaff - sanders model and analytical approximate formulae for bond prices were considered . the coefficients of the taylor expansion of the bond prices and their logarithms in a general one factor model with time independent parameters can be computed recursively and expressed in a closed form . since the prices are expanded into series around , the most precise resuts are expected to be obtained ofr small . however , numerical examples dealing with cox - ingersoll - ross and dothan models show that we are able to obtain very precise approximations also for higher values of . this research was supported by vega 1/0747/12 grant .


In [20]:
train_document['abstract'][test_ind]

'we consider a general one - factor short rate model , in which the instantaneous interest rate is driven by a univariate diffusion with time independent drift and volatility . \n we construct recursive formula for the coefficients of the taylor expansion of the bond price and its logarithm around @xmath0 , where @xmath1 is time to maturity . \n we provide numerical examples of convergence of the partial sums of the series and compare them with the known exact values in the case of cox - ingersoll - ross and dothan model .'

In [21]:
train_document['article'][test_ind]

'interest rate is a rate charged for the use of the money . as an example we show the evolution of several interest rates on the market in figure [ fig : intro ] above . figure [ fig : intro ] below displays the interest rates with different maturities ( so called term structures ) at a given day . short rate models are based on specifying the evolution of the instantaneous interest rates ( also called short rate ) in terms of a stochastic differential equation . the remaining interest rates are then determined by the bond prices . after specifying the so called market price of risk , the price of a bond is a solution to the partial differential equation ( pde ) . alternatively , the short rate evolution can be given in so called risk neutral measure . in particular , if the short rate evolves in the risk neutral measure according to the stochastic differential equation where is a wiener process , then the price of the zero - coupon bond at time when the current level of the short rate

# Cross Attention

In [ ]:
def get_token_score_pairs(article, path = r"bart_finetuned\checkpoint-7746"):
    """"
    This function gives the importance score of each token. 
    Be mindful and use sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    article: str
        the article we want to get the a summary from
    
    path: path
        path to the folder of pre trained model

    Returns
    -------
    token_score_pairs: list of tuples
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #prioritze cuda
    model = BartForConditionalGeneration.from_pretrained(path, attn_implementation="eager").to(device)
    tokenizer = BartTokenizer.from_pretrained(path)

    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    )

    inputs = {k: v.to(device) for k, v in inputs.items()} #makes all inputs set to cuda if possible otherwise cpu

    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True
    )

    cross_attentions = outputs.cross_attentions #gets the cross attention scores for each token in the article
    last_layer = cross_attentions[-1] #looks at last layer
    avg_heads = last_layer.mean(dim=1) #gets the average of all heads of attention scores for each token 
    token_importance = avg_heads.mean(dim=1) #gives overall importance of each token

    input_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids) #gives the tokens
    
    token_scores = [(token, float(score.detach())) for token, score in zip(tokens, token_importance[0])] #gives the list of tuples with tokens and importance score

    return token_scores
    

In [ ]:
def rank_important_lines(token_scores, average_score = False):
    """"
    This function ranks lines based on their importance divided by the amount of tokens
    This only works if we use a sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    token_scores: list of tuples (token, score)
        token_scores is a list of tuples where each tuple contains a token and its corresponding importance score
    
    average_score: bool (default False)
        if True then it will rank based on the average score of all tokens in a line
        if False then it will rank based on the sum of all tokens in a line

    Returns
    -------
    list of tuples (line, total_score, avg_score)
        this function returns a list of tuples where each
    """
    lines = []
    current_line = []
    current_score = 0

    for token, score in token_scores:
        if token != "Ċ": #this token symbolizes new line so if it isn't a new line keeping adding tokens
            current_line.append(token)
            current_score += score
        else:
            if current_line != []: #check if current line is empty or not
                avg_score = current_score / len(current_line) #calculate average score for the line
                lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score, avg_score)) 
            current_line = []
            current_score = 0

    if current_line != []:
        avg_score = current_score / len(current_line) #calculate average score for the line
        lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score, avg_score))

    lines[0]  = (lines[0][0].replace("<s>", "").strip(),  lines[0][1], lines[0][2]) #take out starting token
    lines[-1] = (lines[-1][0].replace("</s>", "").strip(), lines[-1][1], lines[-1][2]) #take out ending token

    if average_score:
        ranked_lines = sorted(lines, key=lambda x: x[2], reverse=True) #ranks the lines based on the average score in desc order
    else:
        ranked_lines = sorted(lines, key=lambda x: x[1], reverse=True) #ranks the lines based on the total score in desc order

    return ranked_lines

In [44]:
pairs = get_token_score_pairs(train_section['article'][3])
rank_important_lines(pairs, average_score = False)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1221.74it/s, Materializing param=model.shared.weight]                                   


[('the interest rates is then given by the formula see , e . g . , or for more details on interest rates modelling',
  0.3838520050048828,
  0.014763538654033955),
 ('solution to the bond pricing p de ( [ eq : p de 1 ] ) is known only i special cases , such as vas ice k model or co x - ing ers oll - ro ss ( cir ) model . sometimes , there is a closed form solution known but it is complicated , for example the bond price in d oth an model involves special functions and two - dimensional integration . in general ,',
  0.04972696304321289,
  0.0006139131239902826),
 ('in particular , if the short rate evolves in the risk neutral measure according to the sto ch astic differential equation where is a wi ener process , then the price of the zero - coupon bond at time when the current level of the short rate is , is a solution to the p de the p de ( [ eq : p de 1 ] ) holds for all and , where is the maturity of the bond',
  0.047715187072753906,
  0.0005818925252774866),
 ('ste hl kov , b ., 

In [22]:
print(summary)

a simple analytic approximation formula for the bond price in the chan - karolyi - longstaff - sanders model and analytical approximate formulae for bond prices were considered . the coefficients of the taylor expansion of the bond prices and their logarithms in a general one factor model with time independent parameters can be computed recursively and expressed in a closed form . since the prices are expanded into series around , the most precise resuts are expected to be obtained ofr small . however , numerical examples dealing with cox - ingersoll - ross and dothan models show that we are able to obtain very precise approximations also for higher values of . this research was supported by vega 1/0747/12 grant .


In [16]:
train_document['abstract'][3]

'we consider a general one - factor short rate model , in which the instantaneous interest rate is driven by a univariate diffusion with time independent drift and volatility . \n we construct recursive formula for the coefficients of the taylor expansion of the bond price and its logarithm around @xmath0 , where @xmath1 is time to maturity . \n we provide numerical examples of convergence of the partial sums of the series and compare them with the known exact values in the case of cox - ingersoll - ross and dothan model .'

In [17]:
train_document['article'][3]

'interest rate is a rate charged for the use of the money . as an example we show the evolution of several interest rates on the market in figure [ fig : intro ] above . figure [ fig : intro ] below displays the interest rates with different maturities ( so called term structures ) at a given day . short rate models are based on specifying the evolution of the instantaneous interest rates ( also called short rate ) in terms of a stochastic differential equation . the remaining interest rates are then determined by the bond prices . after specifying the so called market price of risk , the price of a bond is a solution to the partial differential equation ( pde ) . alternatively , the short rate evolution can be given in so called risk neutral measure . in particular , if the short rate evolves in the risk neutral measure according to the stochastic differential equation where is a wiener process , then the price of the zero - coupon bond at time when the current level of the short rate